# 05 — Analysis & Results

Cross-scenario comparison, DQN vs PPO vs Recurrent PPO head-to-head,
statistical analysis over multiple seeds, qualitative saliency analysis,
and gameplay video generation. Reads training artifacts produced by
notebooks `02_dqn_training.ipynb`, `03_ppo_training.ipynb`, and
`04_recurrent_ppo_training.ipynb`.

## 1. Setup

In [ ]:
# --- Colab Setup ---
# Uncomment the block below when running on Google Colab
# import subprocess, os
# if not os.path.exists("/content/rl-doom"):
#     subprocess.run(["git", "clone", "https://github.com/kuds/rl-doom.git", "/content/rl-doom"], check=True)
# os.chdir("/content/rl-doom/notebooks")
# subprocess.run(["pip", "install", "-q", "-e", "/content/rl-doom[notebooks]"], check=True)

import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from pathlib import Path

from rl_doom.evaluate import evaluate_agent, load_run, record_episode
from rl_doom.paths import (
    latest_run,
    per_experiment_analysis_dir,
    cross_experiment_analysis_dir,
    saliency_analysis_dir,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the block below when running on Google Colab
# ---
# import shutil
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ROOT = "/content/drive/MyDrive/Finding Theta/rl-doom"
# os.makedirs(DRIVE_ROOT, exist_ok=True)
# for subdir in ["training_jobs", "analysis"]:
#     drive_dir = f"{DRIVE_ROOT}/{subdir}"
#     local_dir = os.path.abspath(f"../{subdir}")
#     os.makedirs(drive_dir, exist_ok=True)
#     if os.path.islink(local_dir):
#         os.remove(local_dir)
#     if os.path.isdir(local_dir):
#         for f in os.listdir(local_dir):
#             src = os.path.join(local_dir, f)
#             dst = os.path.join(drive_dir, f)
#             if not os.path.exists(dst):
#                 shutil.move(src, dst)
#         shutil.rmtree(local_dir)
#     os.symlink(drive_dir, local_dir)
# print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")
# ---

In [ ]:
from itertools import product

from rl_doom.env import make_wrapped_env

# Thin passthrough so every call site in this notebook shares the
# wrapper pipeline defined in ``rl_doom.env.make_wrapped_env``.
def make_env(scenario, seed=None):
    return make_wrapped_env(scenario)

# DQN, PPO, and Recurrent PPO are trained on the same four scenarios,
# so we analyse the full Cartesian product.
SCENARIOS = ["basic", "deadly_corridor", "defend_the_center", "deathmatch"]
ALGOS = ["dqn", "ppo", "recurrent_ppo"]
EXPERIMENTS = [(scenario, algo) for scenario, algo in product(SCENARIOS, ALGOS)]

# Resolve the latest run for each (env, algo) experiment. Skip experiments
# whose ``latest`` pointer doesn't exist yet — that lets the notebook run
# even if (e.g.) the recurrent_ppo notebook hasn't been executed for every
# scenario yet, instead of crashing in the loaders below.
RUN_DIRS = {}
for key in EXPERIMENTS:
    try:
        RUN_DIRS[key] = latest_run(*key)
    except Exception as exc:
        print(f"  [skip] {key}: no latest run ({exc})")
EXPERIMENTS = list(RUN_DIRS.keys())
for (env_name, algo), run_dir in RUN_DIRS.items():
    print(f"{env_name:20s}  {algo:14s}  ->  {run_dir}")

## 2. Load trained agents

In [ ]:
# Load every trained checkpoint from its ``latest`` run. ``load_run``
# auto-detects Stable-Baselines3 checkpoints (``final.zip``) and falls
# back to the legacy hand-rolled agent format (``final.pt``), so the
# same call works for both DQN and PPO regardless of which pipeline
# generation wrote the run.
agents = {
    key: load_run(RUN_DIRS[key], device=str(device))
    for key in EXPERIMENTS
}
for (scenario, algo) in EXPERIMENTS:
    print(f"  loaded {algo.upper():3s} on {scenario}")
print(f"{len(agents)} checkpoints loaded.")

## 3. Cross-scenario comparison table

In [ ]:
N_EVAL = 50

results = []

# Evaluate every (scenario, algo) combination so the comparison table
# covers the full matrix, not just the previously-curated one-algo-per-
# scenario selection.
configs = [
    (algo.upper(), scenario, agents[(scenario, algo)])
    for (scenario, algo) in EXPERIMENTS
]

for algo, scenario, agent in configs:
    rews = evaluate_agent(agent, lambda s=scenario: make_env(s), n_episodes=N_EVAL)
    results.append({
        "Algorithm": algo,
        "Scenario": scenario,
        "Mean Reward": np.mean(rews),
        "Std": np.std(rews),
        "Min": np.min(rews),
        "Max": np.max(rews),
        "Median": np.median(rews),
        "N_Episodes": N_EVAL,
    })

df = pd.DataFrame(results)

# Save to CSV for downstream analysis
ce_dir = cross_experiment_analysis_dir()
df.to_csv(ce_dir / "tables" / "cross_scenario_comparison.csv", index=False)
print(f"Saved cross-scenario comparison to {ce_dir / 'tables' / 'cross_scenario_comparison.csv'}")
df

## 4. DQN vs PPO vs Recurrent PPO head-to-head (all scenarios)

For each scenario, plot every algorithm's training curves on a shared
axis so the head-to-head comparison is apples-to-apples. Algorithms with
no `latest` run for a given scenario are silently skipped (so the plot
still renders if you've only trained a subset).

In [ ]:
# Plot DQN, PPO, and Recurrent PPO training curves for every scenario.
# Each row is one scenario: left panel = smoothed episode rewards, right
# panel = smoothed episode lengths. All algorithms are overlaid on the
# same axes per panel.
def _smoothed(series, window=20):
    series = np.asarray(series)
    if len(series) < window:
        return None
    return np.convolve(series, np.ones(window) / window, mode="valid")

ALGO_COLORS = {
    "dqn": "tab:blue",
    "ppo": "tab:orange",
    "recurrent_ppo": "tab:green",
}
ALGO_LABELS_PLOT = {
    "dqn": "DQN",
    "ppo": "PPO",
    "recurrent_ppo": "Recurrent PPO",
}

missing = []
fig, axes = plt.subplots(len(SCENARIOS), 2, figsize=(14, 4 * len(SCENARIOS)))
if len(SCENARIOS) == 1:
    axes = np.array([axes])

for row, scenario in enumerate(SCENARIOS):
    reward_ax = axes[row, 0]
    length_ax = axes[row, 1]

    for algo in ALGOS:
        if (scenario, algo) not in RUN_DIRS:
            continue
        color = ALGO_COLORS[algo]
        label = ALGO_LABELS_PLOT[algo]
        metrics_path = RUN_DIRS[(scenario, algo)] / "metrics" / "training.npz"
        if not metrics_path.exists():
            missing.append(metrics_path)
            continue
        data = np.load(metrics_path, allow_pickle=True)

        rewards = data["episode_rewards"]
        reward_ax.plot(rewards, alpha=0.25, color=color)
        sm = _smoothed(rewards)
        if sm is not None:
            reward_ax.plot(range(19, 19 + len(sm)), sm, color=color, label=f"{label} MA-20")

        if "episode_lengths" in data:
            lengths = data["episode_lengths"]
            length_ax.plot(lengths, alpha=0.25, color=color)
            sm = _smoothed(lengths)
            if sm is not None:
                length_ax.plot(range(19, 19 + len(sm)), sm, color=color, label=f"{label} MA-20")

    reward_ax.set_title(f"{scenario} — Rewards")
    reward_ax.set_xlabel("Episode")
    reward_ax.set_ylabel("Reward")
    reward_ax.legend()

    length_ax.set_title(f"{scenario} — Episode length")
    length_ax.set_xlabel("Episode")
    length_ax.set_ylabel("Steps")
    length_ax.legend()

plt.suptitle("DQN vs PPO vs Recurrent PPO — Training curves per scenario", fontsize=14)
plt.tight_layout()
plt.savefig(ce_dir / "figures" / "algo_comparison_training.png", dpi=150, bbox_inches="tight")
plt.show()

if missing:
    print("Training metrics not found for the following runs (re-run notebooks 02/03/04 to generate):")
    for p in missing:
        print(f"  {p}")

## 5. Statistical analysis — multiple seeds

Run evaluation with different seeds to compute mean ± std with confidence intervals.

In [ ]:
N_SEEDS = 5
EPISODES_PER_SEED = 20

def multi_seed_eval(agent, scenario, n_seeds=N_SEEDS, episodes=EPISODES_PER_SEED):
    """Evaluate ``agent`` across ``n_seeds`` seeds, ``episodes`` each.

    Uses :func:`rl_doom.evaluate.evaluate_agent`, which dispatches on the
    agent's API — works transparently for legacy DQNAgent/PPOAgent and
    for Stable-Baselines3 models that expose ``.predict()``.
    """
    seed_means = []
    all_rewards = []
    for seed in range(n_seeds):
        # evaluate_agent builds its own env from the factory each call.
        rews = evaluate_agent(
            agent, lambda s=scenario: make_env(s), n_episodes=episodes,
        )
        seed_means.append(float(rews.mean()))
        all_rewards.extend(rews.tolist())
    return np.array(seed_means), np.array(all_rewards)

multi_seed_results = []
for algo, scenario, agent in configs:
    means, all_rews = multi_seed_eval(agent, scenario)
    stderr = means.std() / np.sqrt(len(means))
    ci_95 = 1.96 * stderr
    row = {
        "Algorithm": algo,
        "Scenario": scenario,
        "Grand Mean": f"{means.mean():.2f}",
        "Std (seeds)": f"{means.std():.2f}",
        "95% CI": f"+/- {ci_95:.2f}",
        "Per-seed means": means.tolist(),
    }
    multi_seed_results.append(row)
    print(
        f"{algo:4s} | {scenario:20s} | "
        f"Mean: {means.mean():.2f} +/- {ci_95:.2f} (95% CI)  "
        f"Std: {means.std():.2f}  Seeds: {[f'{m:.1f}' for m in means]}"
    )

multi_seed_df = pd.DataFrame(multi_seed_results)
multi_seed_df.to_csv(ce_dir / "tables" / "multi_seed_evaluation.csv", index=False)
print(f"\nSaved multi-seed evaluation to {ce_dir / 'tables' / 'multi_seed_evaluation.csv'}")

## 6. Qualitative analysis — saliency maps

Compute input-gradient saliency to see which pixels the agent focuses
on. Restricted to **DQN and PPO** here: Recurrent PPO's policy needs
LSTM hidden state threaded through every forward pass, so a clean
single-frame gradient isn't well-defined without also fixing a hidden
state — left as an exercise for the curious reader.

In [ ]:
def compute_saliency(agent, obs):
    """Gradient-based saliency map. Supports both legacy agents and SB3 models.

    * Legacy DQN :            ``agent.policy_net(obs)``
    * Legacy PPO :            ``agent.network(obs)`` -> (logits, value)
    * SB3 DQN   :             ``agent.q_net(obs)``
    * SB3 PPO   :             ``agent.policy.get_distribution(obs).distribution.logits``
    """
    obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
    obs_t.requires_grad_(True)

    if hasattr(agent, "policy_net"):           # legacy DQN
        score = agent.policy_net(obs_t).max()
    elif hasattr(agent, "network"):            # legacy PPO
        logits, _ = agent.network(obs_t)
        score = logits.max()
    elif hasattr(agent, "q_net"):              # SB3 DQN
        score = agent.q_net(obs_t).max()
    elif hasattr(agent, "policy"):             # SB3 PPO / any actor-critic
        dist = agent.policy.get_distribution(obs_t)
        score = dist.distribution.logits.max()
    else:
        raise TypeError(f"Unsupported agent type for saliency: {type(agent).__name__}")

    score.backward()
    saliency = obs_t.grad.data.abs().squeeze().cpu().numpy()
    return saliency.mean(axis=0)  # average over stacked frames

In [ ]:
# Compute saliency for a few states. We use the Basic scenario (stable,
# well-learned policy for both algos) and compare DQN and PPO attention
# on the same rollout.
env_sal = make_env("basic")
obs_sal, _ = env_sal.reset(seed=42)

agents_for_saliency = [
    ("dqn", agents[("basic", "dqn")]),
    ("ppo", agents[("basic", "ppo")]),
]

fig, axes = plt.subplots(len(agents_for_saliency) + 1, 4, figsize=(16, 4 * (len(agents_for_saliency) + 1)))
saliency_maps = {algo: [] for algo, _ in agents_for_saliency}
observation_frames = []

# Use the DQN agent to drive the rollout so every saliency map is
# computed on the same sequence of observations.
driver_agent = agents[("basic", "dqn")]
for i in range(4):
    for _ in range(i * 5 + 1):
        action, _ = driver_agent.predict(obs_sal, deterministic=True)
        obs_sal, _, term, trunc, _ = env_sal.step(int(np.asarray(action).flatten()[0]))
        if term or trunc:
            obs_sal, _ = env_sal.reset(seed=42 + i)

    observation_frames.append(obs_sal[-1])
    axes[0, i].imshow(obs_sal[-1], cmap="gray")
    axes[0, i].set_title(f"Observation {i}")
    axes[0, i].axis("off")

    for row, (algo, agent) in enumerate(agents_for_saliency, start=1):
        saliency = compute_saliency(agent, obs_sal)
        saliency_maps[algo].append(saliency)
        axes[row, i].imshow(saliency, cmap="hot")
        axes[row, i].set_title(f"{algo.upper()} saliency {i}")
        axes[row, i].axis("off")

plt.suptitle("DQN vs PPO Saliency Maps — Basic Scenario", fontsize=14)
plt.tight_layout()
sal_dir = saliency_analysis_dir()
plt.savefig(sal_dir / "basic_saliency.png", dpi=150, bbox_inches="tight")
plt.show()
env_sal.close()

np.savez(
    sal_dir / "basic_saliency.npz",
    observation_frames=np.array(observation_frames),
    **{f"{algo}_saliency": np.array(maps) for algo, maps in saliency_maps.items()},
)
print(f"Saved saliency arrays to {sal_dir / 'basic_saliency.npz'}")

## 7. Gameplay video generation

In [ ]:
from rl_doom.evaluate import record_and_save_video

# Save one gameplay video per trained agent. ``load_run(..., prefer="best")``
# pulls the best eval checkpoint saved by SB3's ``EvalCallback`` so the
# clip showcases the top-performing weights rather than the final-step
# snapshot (which may have drifted below peak after late-training updates).
# ``record_and_save_video`` writes an MP4 by default, with an automatic
# GIF fallback when ``imageio-ffmpeg`` is not available.
video_paths = {}
for name, scenario, _ in configs:
    best_agent = load_run(
        RUN_DIRS[(scenario, name.lower())],
        device=str(device),
        prefer="best",
    )
    pe_dir = per_experiment_analysis_dir(scenario, name.lower())
    media_dir = pe_dir / "media"
    video_path = media_dir / f"{name.lower()}_{scenario}.mp4"
    video_path = record_and_save_video(
        best_agent,
        lambda s=scenario: make_env(s),
        video_path,
        fps=20,
    )
    video_paths[(name, scenario)] = video_path
    print(f"Saved {video_path}")

In [ ]:
from IPython.display import Video, display

for name, scenario, _ in configs:
    video_path = video_paths.get((name, scenario))
    if video_path is None or not video_path.exists():
        continue
    print(f"\n{name} — {scenario}: {video_path.name}")
    display(Video(str(video_path), embed=True))

## 8. Summary

The code cell below builds the final comparison table **from the CSVs
written earlier in this notebook** (`cross_scenario_comparison.csv` and
`multi_seed_evaluation.csv`), so the numbers stay in lock-step with
whatever training runs are currently pointed at by `latest`. It writes a
shareable Markdown summary to
`analysis/cross_experiment/cross_scenario_summary.md` and renders it
inline.

### Artifacts saved
- `../analysis/cross_experiment/tables/cross_scenario_comparison.csv` — evaluation results table (every algorithm × scenario)
- `../analysis/cross_experiment/tables/multi_seed_evaluation.csv` — multi-seed statistical analysis with 95% CIs
- `../analysis/cross_experiment/cross_scenario_summary.md` — rendered Markdown summary (generated below)
- `../analysis/cross_experiment/figures/algo_comparison_training.png` — DQN vs PPO vs Recurrent PPO training curves for every scenario
- `../analysis/saliency/basic_saliency.{png,npz}` — DQN + PPO saliency maps on the Basic scenario
- `../analysis/per_experiment/<env>/<algo>/media/*.mp4` — gameplay videos (MP4, GIF fallback)

In [ ]:
from IPython.display import Markdown, display

# Re-load the CSVs from section 3 and 5 so the summary always matches the
# most recently generated artifacts (no risk of stale hand-edited numbers).
cmp_path = ce_dir / "tables" / "cross_scenario_comparison.csv"
ms_path = ce_dir / "tables" / "multi_seed_evaluation.csv"
cmp_df = pd.read_csv(cmp_path)
ms_df = pd.read_csv(ms_path)

# Pull training budgets straight from each run's ``config.json`` so the
# ``Training steps`` column always matches what was actually run.
import json

def _training_steps(run_dir):
    cfg = json.loads((run_dir / "config.json").read_text())
    hp = cfg.get("hyperparams", {})
    return int(hp.get("total_timesteps") or hp.get("total_steps") or 0)

steps = {
    (scenario, algo): _training_steps(RUN_DIRS[(scenario, algo)])
    for (scenario, algo) in EXPERIMENTS
}

# Merge single-shot eval (mean/std from section 3) with multi-seed stats
# (grand mean + 95% CI from section 5). The multi-seed CSV stores the
# per-seed means as a stringified list, so we re-parse it with literal_eval
# to avoid an eval() injection footgun.
import ast

rows = []
for (scenario, algo) in EXPERIMENTS:
    algo_upper = algo.upper()
    cmp_row = cmp_df[(cmp_df["Algorithm"] == algo_upper) & (cmp_df["Scenario"] == scenario)]
    ms_row = ms_df[(ms_df["Algorithm"] == algo_upper) & (ms_df["Scenario"] == scenario)]
    if cmp_row.empty or ms_row.empty:
        continue
    mean = float(cmp_row["Mean Reward"].iloc[0])
    std = float(cmp_row["Std"].iloc[0])
    try:
        per_seed = np.array(ast.literal_eval(str(ms_row["Per-seed means"].iloc[0])), dtype=float)
        ci = 1.96 * per_seed.std() / np.sqrt(len(per_seed)) if len(per_seed) else float("nan")
    except (ValueError, SyntaxError):
        ci = float("nan")
    rows.append({
        "Algorithm": algo_upper,
        "Scenario": scenario,
        "Mean reward": f"{mean:.2f}",
        "Std": f"{std:.2f}",
        "95% CI": "" if np.isnan(ci) else f"±{ci:.2f}",
        "Training steps": f"{steps[(scenario, algo)]:,}" if steps[(scenario, algo)] else "—",
    })

summary_df = pd.DataFrame(rows)
md_table = summary_df.to_markdown(index=False)
summary_path = ce_dir / "cross_scenario_summary.md"
summary_path.write_text(
    "# Cross-scenario summary\n\n"
    "Auto-generated from `tables/cross_scenario_comparison.csv` and "
    "`tables/multi_seed_evaluation.csv`.\n\n"
    + md_table + "\n"
)
print(f"Saved cross-scenario summary to {summary_path}")
display(Markdown(md_table))

In [ ]:
# Disconnect the Colab runtime at the end of the notebook to save compute.
# No-op when running locally.
try:
    from google.colab import runtime as _colab_runtime
except ImportError:
    pass
else:
    import time
    print("Notebook finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    _colab_runtime.unassign()